In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics.pairwise import cosine_similarity

# Task 1: Exploring the Dataset with Implicit Feedback

## Step 1: Load the Data

In [ ]:
# Charger les fichiers
interactions = pd.read_csv('https://raw.githubusercontent.com/Vane-s/UNIL_Rolex/main/interactions_train.csv')
items = pd.read_csv("https://raw.githubusercontent.com/Vane-s/UNIL_Rolex/main/items.csv")

# Afficher les premières lignes de chaque ensemble de données
display(interactions.head())
display(items.head())

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


,Title,Author,ISBN Valid,Publisher,Subjects,i
0,Classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,Classification décimale universelle; Indexatio...,0
1,Les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,Histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,Histoires de vie en sociologie; Sciences socia...,2
3,Ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,Moyen-Orient; Bandes dessinées autobiographiqu...,3
4,Les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,France--1945-1975; Roman historique; Roman fra...,4


## Step 2 : Clean the data

In [ ]:
# Renommer les colonnes du dataset interactions
interactions.rename(columns={
    'u': 'user_id',        # Colonne utilisateur
    'i': 'item_id',        # Colonne identifiant du livre
    't': 'timestamp'       # Colonne horodatage
}, inplace=True)

# Renommer les colonnes du dataset items
items.rename(columns={
    'Title': 'title',          # Titre du livre
    'Author': 'author',        # Auteur
    'ISBN Valid': 'isbn',      # ISBN
    'Publisher': 'publisher',  # Éditeur
    'Subjects': 'subjects',    # Catégories/thèmes
    'i': 'item_id'             # Identifiant du livre
}, inplace=True)

# Vérifier les colonnes après renommage
print("Colonnes interactions:", interactions.columns)
print("Colonnes items:", items.columns)


Colonnes interactions: Index(['user_id', 'item_id', 'timestamp'], dtype='object')
Colonnes items: Index(['title', 'author', 'isbn', 'publisher', 'subjects', 'item_id'], dtype='object')


In [ ]:
# Nettoyage des interactions
# Supprimer les doublons ayant exactement le même user_id, item_id et timestamp
interactions.drop_duplicates(subset=['user_id', 'item_id', 'timestamp'], inplace=True)
print("Nombre d'interactions après suppression des doublons:", len(interactions))

Nombre d'interactions après suppression des doublons: 87045


In [ ]:
# Convertir les colonnes en types appropriés
interactions['user_id'] = interactions['user_id'].astype(str)
interactions['item_id'] = interactions['item_id'].astype(str)
interactions['timestamp'] = pd.to_datetime(interactions['timestamp'], unit='s', errors='coerce')
print("Types de données après conversion:", interactions.dtypes)

Types de données après conversion: user_id              object
item_id              object
timestamp    datetime64[ns]
dtype: object


In [ ]:
# Supprimer les entrées avec des timestamps invalides
interactions.dropna(subset=['timestamp'], inplace=True)
print("Nombre d'interactions après suppression des timestamps invalides:", len(interactions))

Nombre d'interactions après suppression des timestamps invalides: 87045


In [ ]:
# Supprimer les lignes avec des valeurs manquantes dans les colonnes critiques
items.dropna(subset=['item_id', 'title'], inplace=True)

In [ ]:
# Standardiser les champs textuels (par exemple, les titres, sujets)
items['title'] = items['title'].str.strip().str.lower()
items['subjects'] = items['subjects'].str.strip().str.lower() if 'subjects' in items.columns else None
print("Exemples de titres après standardisation:", items['title'].head())
print("Exemples de sujets après standardisation:", items['subjects'].head() if 'subjects' in items.columns else "N/A")

Exemples de titres après standardisation: 0    classification décimale universelle : édition ...
1    les interactions dans l'enseignement des langu...
2    histoire de vie et recherche biographique : pe...
3    ce livre devrait me permettre de résoudre le c...
4                      les années glorieuses : roman /
Name: title, dtype: object
Exemples de sujets après standardisation: 0    classification décimale universelle; indexatio...
1    didactique--langue étrangère - enseignement; d...
2    histoires de vie en sociologie; sciences socia...
3    moyen-orient; bandes dessinées autobiographiqu...
4    france--1945-1975; roman historique; roman fra...
Name: subjects, dtype: object


In [ ]:
# Sauvegarder les fichiers CSV dans l'environnement Colab
interactions_file = "cleaned_interactions.csv"
interactions.to_csv(interactions_file, index=False)

items_file = "cleaned_items.csv"
items.to_csv(items_file, index=False)

# Télécharger le fichier localement
from google.colab import files
files.download(interactions_file)
files.download(items_file)


print("Nettoyage terminé. Les fichiers nettoyés ont été enregistrés.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Nettoyage terminé. Les fichiers nettoyés ont été enregistrés.


In [ ]:
# Afficher les premières lignes de chaque ensemble de données nettoyés
display(interactions.head())
display(items.head())
print(interactions.shape)
print(items.shape)

,user_id,item_id,timestamp
0,4456,8581,2023-06-23 17:24:46
1,142,1964,2023-03-23 15:30:06
2,362,3705,2024-02-02 11:00:59
3,1809,11317,2023-01-12 14:19:22
4,4384,1323,2023-04-13 16:09:22


,title,author,isbn,publisher,subjects,item_id
0,classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,classification décimale universelle; indexatio...,0
1,les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,histoires de vie en sociologie; sciences socia...,2
3,ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,moyen-orient; bandes dessinées autobiographiqu...,3
4,les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,france--1945-1975; roman historique; roman fra...,4


(87045, 3)
(15291, 6)


## Step 2: Check the Number of interactions, users and books

In [ ]:
n_users = interactions.user_id.nunique()
n_items = interactions.item_id.nunique()  # Livres avec interactions
n_items_total = items.item_id.nunique()  # Livres totaux dans la base "items"
print(f"Number of users = {n_users}, \n Number of books with interactions = {n_items} \n Total books in dataset = {n_items_total} \n Number of interactions = {len(interactions)}")

Number of users = 7838, 
 Number of books with interactions = 15109 
 Total books in dataset = 15291 
 Number of interactions = 87045


## Step 3: Split the Data into Training and Test Sets

In [ ]:
# trions d'abord les interactions par utilisateur et par date.
interactions = interactions.sort_values(["user_id", "timestamp"])
interactions.head()

,user_id,item_id,timestamp
21035,0,0,2023-03-30 15:44:30
28842,0,1,2023-04-06 12:13:54
3958,0,2,2023-04-06 17:15:08
29592,0,3,2023-05-10 10:35:45
6371,0,3,2023-05-10 10:35:50


In [ ]:
#Calcul du rang relatif des interactions par utilisateur
interactions["pct_rank"] = interactions.groupby("user_id")["timestamp"].rank(pct=True, method='dense')
interactions.reset_index(inplace=True, drop=True)
interactions.head()

train_data = interactions[interactions["pct_rank"] < 0.8]
test_data = interactions[interactions["pct_rank"] >= 0.8]
print("Training set size:", train_data.shape[0])
print("Testing set size:", test_data.shape[0])

Training set size: 65417
Testing set size: 21628


# Task 2: Creating User-Item Matrices for Implicit Feedback

In [ ]:
print('Number of users =', n_users, '| Number of movies =', n_items)

Number of users = 7838 | Number of movies = 15109


## Step 1: Define the Function to Create the Data Matrix

In [ ]:
import numpy as np

# Define a function to create the data matrix
def create_data_matrix(data, n_users, n_items):
    """
    This function returns a numpy matrix with shape (n_users, n_items).
    Each entry is a binary value indicating positive interaction.
    """
    data_matrix = np.zeros((n_users, n_items))
    # Convert 'user_id' and 'item_id' to integers before using them as indices
    data_matrix[data["user_id"].astype(int).values, data["item_id"].astype(int).values] = 1
    return data_matrix

## Step 2: Create the Training and Testing Matrices

In [ ]:
# Créer les matrices d'apprentissage et de test
train_data_matrix = create_data_matrix(train_data, n_users, n_items_total)
test_data_matrix = create_data_matrix(test_data, n_users, n_items_total)

# Afficher les matrices pour comprendre leur structure
print('train_data_matrix')
print(train_data_matrix)
print("number of non-zero values: ", np.sum(train_data_matrix))
print('test_data_matrix')
print(test_data_matrix)
print("number of non-zero values: ", np.sum(test_data_matrix))

train_data_matrix
[[1. 1. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
number of non-zero values:  49689.0
test_data_matrix
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
number of non-zero values:  19409.0


# Task 3: Item-to-Item Collaborative Filtering with Implicit Feedback

## Step 1: Compute Item Similarity Matrix

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculer la matrice de similarité élément-élément
# utiliser cette fonction
item_similarity = cosine_similarity(train_data_matrix.T)
print("Item-Item Similarity Matrix:")
print(item_similarity)
print(item_similarity.shape)

Item-Item Similarity Matrix:
[[1.         0.40824829 0.33333333 ... 0.         0.         0.        ]
 [0.40824829 1.         0.40824829 ... 0.         0.         0.        ]
 [0.33333333 0.40824829 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]]
(15291, 15291)


## Step 2: Predict Positive Interactions Using Item Similarity

In [ ]:
import numpy as np

# Define the function to predict interactions based on item similarity
def item_based_predict(interactions, similarity, epsilon=1e-9):
    """
    Predicts user-item interactions based on item-item similarity.
    Parameters:
        interactions (numpy array): The user-item interaction matrix.
        similarity (numpy array): The item-item similarity matrix.
        epsilon (float): Small constant added to the denominator to avoid division by zero.
    Returns:
        numpy array: The predicted interaction scores for each user-item pair.
    """
    # np.dot does the matrix multiplication. Here we are calculating the
    # weighted sum of interactions based on item similarity
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T  # Transpose to get users as rows and items as columns

# Calculate the item-based predictions for positive interactions
item_prediction = item_based_predict(train_data_matrix, item_similarity)
print("Predicted Interaction Matrix:")
print(item_prediction)
print(item_prediction.shape)

Predicted Interaction Matrix:
[[0.46097255 0.85435984 0.71867834 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
(7838, 15291)


# Task 4: User-to-User Collaborative Filtering with Implicit Feedback

## Step 1: Compute User Similarity Matrix

In [ ]:
# Compute the user-user similarity matrix
user_similarity = cosine_similarity(train_data_matrix)
print("User-User Similarity Matrix:")
print(user_similarity)

# Check the shape as a sanity check
print("Shape of User Similarity Matrix:", user_similarity.shape)

User-User Similarity Matrix:
[[1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]
Shape of User Similarity Matrix: (7838, 7838)


## Step 2: Predict Positive Interactions Using User Similarity

In [ ]:
# Define the function to predict interactions based on user similarity
def user_based_predict(interactions, similarity, epsilon=1e-9):
    """
    Predicts user-item interactions based on user-user similarity.
    Parameters:
        interactions (numpy array): The user-item interaction matrix.
        similarity (numpy array): The user-user similarity matrix.
        epsilon (float): Small constant added to the denominator to avoid division by zero.
    Returns:
        numpy array: The predicted interaction scores for each user-item pair.
    """
    # Calculate the weighted sum of interactions based on user similarity
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

# Calculate the user-based predictions for positive interactions
user_prediction = user_based_predict(train_data_matrix, user_similarity)
print("Predicted Interaction Matrix (User-Based):")
print(user_prediction)
print(user_prediction.shape)

Predicted Interaction Matrix (User-Based):
[[0.13171649 0.13259591 0.14274867 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
(7838, 15291)


# Task 5: Evaluating Our Recommenders

In [ ]:
import numpy as np

# TODO: Implement the precision_recall_at_k function
def precision_recall_at_k(prediction, ground_truth, k=10):
    """
    Calculates Precision@K and Recall@K for top-K recommendations.
    Parameters:
        prediction (numpy array): The predicted interaction matrix with scores.
        ground_truth (numpy array): The ground truth interaction matrix (binary).
        k (int): Number of top recommendations to consider.
    Returns:
        precision_at_k (float): The average precision@K over all users.
        recall_at_k (float): The average recall@K over all users.
    """
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):
        # TODO: Get the indices of the top-K items for the user based on predicted scores
        top_k_items = np.argsort(prediction[user, :])[-k:]

        # TODO: Calculate the number of relevant items in the top-K items for the user
        relevant_items_in_top_k = np.isin(top_k_items, np.where(ground_truth[user, :] == 1)[0]).sum()

        # TODO: Calculate the total number of relevant items for the user
        total_relevant_items = ground_truth[user, :].sum()

        # Precision@K and Recall@K for this user
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0

    # Average Precision@K and Recall@K over all users
    precision_at_k /= num_users
    recall_at_k /= num_users

    return precision_at_k, recall_at_k

In [ ]:
# Run this line after implementing the function
precision_user_k, recall_user_k = precision_recall_at_k(user_prediction, test_data_matrix, k=10)
precision_item_k, recall_item_k = precision_recall_at_k(item_prediction, test_data_matrix, k=10)

print('User-based CF Precision@K:', precision_user_k)
print('User-based CF Recall@K:', recall_user_k)
print('Item-based CF Precision@K:', precision_item_k)
print('Item-based CF Recall@K:', recall_item_k)

User-based CF Precision@K: 0.056532278642513596
User-based CF Recall@K: 0.2906567293761024
Item-based CF Precision@K: 0.05561367695841055
Item-based CF Recall@K: 0.26399361388179715


# Task 6: Recommendations for users

In [ ]:
import random
import numpy as np

# Pick a user at random
user_id = range(user_prediction.shape[0])
print("Selected User ID:", user_id)

# Get top-10 recommendations for the user from User-to-User CF
user_top_10 = np.argsort(user_prediction[user_id, :])[-10:][::-1]

# Get top-10 recommendations for the user from Item-to-Item CF
item_top_10 = np.argsort(item_prediction[user_id, :])[-10:][::-1]

# Display top-10 recommended movie IDs for both models
display(user_top_10)
display(item_top_10)

Selected User ID: range(0, 7838)


array([[    0, 10182, 10183, ...,  3377,  2191,    88],
       [    0, 10174, 10175, ...,   611,  3471, 14550],
       [    0, 10145, 10146, ..., 11126,  4820,  3055],
       ...,
       [    0, 10180, 10181, ...,   494, 10364,  1392],
       [    0, 10147, 10148, ...,  4414,  6364,  1826],
       [    0, 10163, 10164, ...,  2876,  3843,  1826]])

array([[    0, 10182, 10183, ...,    88,  2191, 10542],
       [    0, 10174, 10175, ..., 15184, 14557, 14550],
       [    0, 10145, 10146, ...,  6317, 13016, 15247],
       ...,
       [    0, 10180, 10181, ...,  5064,   494, 10364],
       [    0, 10147, 10148, ..., 13223, 11532,  7426],
       [    0, 10163, 10164, ..., 14105,  3843,  3844]])

In [ ]:
# Générer les 10 meilleures prédictions pour chaque utilisateur
recommendations = []
for user_id in range(user_prediction.shape[0]):
    top_10 = np.argsort(user_prediction[user_id, :])[-10:][::-1]
    recommendations.append(" ".join(map(str, top_10)))

# Créer un DataFrame avec les résultats
import pandas as pd #Import pandas to use DataFrames
submission_df = pd.DataFrame({
    "user_id": range(user_prediction.shape[0]),
    "recommendation": recommendations
})

# Sauvegarder le fichier CSV dans l'environnement Colab
submission_file = "predictions_submission.csv"
submission_df.to_csv(submission_file, index=False)

# Télécharger le fichier localement
#Import the files object from google.colab for file downloading
from google.colab import files
files.download(submission_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>